# Gaia DR4 archive access

Test access to `https://gea.esac.esa.int/`, query the `gaiadr4` schema, and
retrieve coordinates, G magnitude and BP−RP for one source. A successful TAP
query establishes data access; a login or HTTP success alone does not.


The data are described in the Gaia DR4 publication by **Wyrzykowski et al. (2026)**.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from io import BytesIO
from getpass import getpass
import gzip, hashlib, json, re, socket, subprocess, time
import xml.etree.ElementTree as ET
import numpy as np
from astropy.table import Column, Table
from astroquery.gaia import GaiaClass
from IPython.display import display

# Endpoint, TAP schema and DataLink release are independent.
ARCHIVE_URL = "https://gea.esac.esa.int/"
AUTHENTICATE = False  # Public access; enable if the endpoint requires login
TAP_SCHEMA = "gaiadr4"
DATA_RELEASE = "Gaia DR4"  # DataLink label; TAP queries use TAP_SCHEMA
GAIA_USERNAME = "YOUR_GAIA_USERNAME"
AUTH_METHOD = "prompt"  # Optional: "keychain" on macOS with your own entry
KEYCHAIN_SERVICE = "gaia-archive"
SMOKE_SOURCE_ID = "704967037090946688"
JOB_WAIT_SECONDS = 1800
NETWORK_TIMEOUT_SECONDS = 120


def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "01_archive_access.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the gaia-dr4-microlensing-notebooks folder.")

ROOT = find_project_root(Path.cwd())

OUTPUT_DIRECTORY = ROOT / "outputs" / TAP_SCHEMA
OUTPUT = OUTPUT_DIRECTORY / "archive"
OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG = dict(endpoint=ARCHIVE_URL, schema=TAP_SCHEMA, data_release=DATA_RELEASE)
print(CONFIG)

GAIA_SOURCE_TABLE = f"{TAP_SCHEMA}.gaia_source"

## Archive helpers and optional authentication

Public access is the default (`AUTHENTICATE = False`). If access requires an
archive account, enable authentication, enter your username, and use the private
password prompt. The optional macOS Keychain mode uses your own `gaia-archive`
entry. Credentials are never written to a notebook or output file, and an
authenticated session always logs out in `finally`.

Each session begins with a fresh known-source TAP query. Requests are sequential;
authenticated mode checks the shared account limits of five executing and 100
queued jobs. ADQL, job IDs, dates, original VOTables and endpoint/schema/release
settings are retained. Timed-out submitted jobs are resumed, and TAP errors or
overflow are rejected. Keep source IDs as exact integers or decimal strings.

In [ ]:
def now():
    return datetime.now(timezone.utc).isoformat()

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2))

def source_id(value):
    if isinstance(value, (float, np.floating)):
        raise ValueError("A floating-point source ID may already have lost precision.")
    value = str(value)
    if not re.fullmatch(r"[0-9]{1,19}", value) or int(value) <= 0:
        raise ValueError("Source IDs must be positive decimal strings or exact integers.")
    return value

def identifier(value):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", value):
        raise ValueError("Invalid ADQL identifier")
    return value

def authenticate(gaia):
    if not GAIA_USERNAME.strip() or GAIA_USERNAME == "YOUR_GAIA_USERNAME":
        raise ValueError("Set GAIA_USERNAME to your own authorised archive account.")
    captured = None
    password = None
    try:
        if AUTH_METHOD == "keychain":
            captured = subprocess.run(
                ["security", "find-generic-password", "-a", GAIA_USERNAME,
                 "-s", KEYCHAIN_SERVICE, "-w"],
                capture_output=True, text=True, check=True, timeout=15)
            password = captured.stdout.rstrip("\n")
        elif AUTH_METHOD == "prompt":
            password = getpass("Gaia archive password: ")
        else:
            raise ValueError("AUTH_METHOD must be prompt or keychain")
        if not password:
            raise ValueError("Empty password")
        gaia.login(user=GAIA_USERNAME, password=password, verbose=False)
    finally:
        del password, captured
        # Astroquery also retains passwords internally; discard those references.
        for client in (gaia, getattr(gaia, "_GaiaClass__gaiadata", None)):
            if client is not None and hasattr(client, "_TapPlus__pwd"):
                client._TapPlus__pwd = None

def checked_votable(path):
    payload = path.read_bytes()
    if payload[:2] == b"\x1f\x8b":
        payload = gzip.decompress(payload)
    root = ET.fromstring(payload)
    statuses = [n.get("value", "").upper() for n in root.iter()
                if n.tag.split("}")[-1] == "INFO" and n.get("name", "").upper() == "QUERY_STATUS"]
    if not statuses or any(s != "OK" for s in statuses):
        raise RuntimeError(f"TAP output is incomplete or failed: QUERY_STATUS={statuses}")
    result = Table.read(BytesIO(payload), format="votable")
    # Normalize archive column names without changing values or units.
    for column in list(result.colnames):
        if column.lower() != column:
            result.rename_column(column, column.lower())
    return result

def tap(gaia, query, name, refresh=False):
    signature = json.dumps(dict(CONFIG, query=query), sort_keys=True)
    key = name + "_" + hashlib.sha256(signature.encode()).hexdigest()[:12]
    folder = OUTPUT / "tap" / key
    folder.mkdir(parents=True, exist_ok=True)
    result_file, ledger = folder / "result.vot", folder / "job.json"
    (folder / "query.adql").write_text(query)
    state = json.loads(ledger.read_text()) if ledger.exists() else dict(CONFIG, query=query, created=now())
    if state.get("status") == "complete":
        if result_file.exists() and not refresh:
            return checked_votable(result_file)
        if refresh:
            # Refresh only a completed query. A submitted/timed-out job is resumed.
            state = dict(CONFIG, query=query, created=now())
    if state.get("job_id"):
        job = gaia.load_async_job(jobid=state["job_id"], load_results=False, verbose=False)
    else:
        jobs = gaia.list_async_jobs(verbose=False) if AUTHENTICATE else []
        phases = [j.get_phase().upper() for j in jobs]
        if phases.count("EXECUTING") >= 5 or phases.count("QUEUED") >= 100:
            raise RuntimeError("Shared account job capacity is full; inspect existing jobs before retrying.")
        job = gaia.launch_job_async(query, name=name, output_file=str(result_file),
            output_format="votable", dump_to_file=True, background=True, verbose=False)
        state.update(job_id=str(job.jobid), status="submitted")
        save_json(ledger, state)  # Persist immediately, before polling.
    deadline = time.monotonic() + JOB_WAIT_SECONDS
    while True:
        phase = job.get_phase(update=True).upper()
        state.update(phase=phase, checked=now())
        save_json(ledger, state)
        if phase == "COMPLETED":
            break
        if phase in {"ERROR", "ABORTED"}:
            raise RuntimeError(f"TAP job {job.jobid} is {phase}; inspect it before replacement.")
        if time.monotonic() > deadline:
            raise TimeoutError(f"Job {job.jobid} retained for resumption; no replacement was submitted.")
        time.sleep(2)
    job.outputFile = str(result_file)
    job.outputFileUser = str(result_file)
    job.save_results(verbose=False)  # Preserve the original server response.
    result = checked_votable(result_file)
    result.write(folder / "result.ecsv", format="ascii.ecsv", overwrite=True)
    state.update(status="complete", rows=len(result), completed=now(), query_status_checked=True)
    save_json(ledger, state)
    return result


@contextmanager
def archive_session():
    identifier(TAP_SCHEMA)
    previous_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(NETWORK_TIMEOUT_SECONDS)
    gaia = None
    try:
        gaia = GaiaClass(gaia_tap_server=ARCHIVE_URL, gaia_data_server=ARCHIVE_URL,
                         show_server_messages=False)
        gaia.ROW_LIMIT = -1
        if AUTHENTICATE:
            authenticate(gaia)
        smoke = tap(gaia, f"SELECT source_id FROM {TAP_SCHEMA}.gaia_source "
                    f"WHERE source_id={source_id(SMOKE_SOURCE_ID)}", "smoke", refresh=True)
        if len(smoke) != 1:
            raise RuntimeError("Known-source TAP test failed; check endpoint, schema and access.")
        yield gaia
    finally:
        try:
            if gaia is not None and AUTHENTICATE:
                gaia.logout(verbose=False)
        finally:
            socket.setdefaulttimeout(previous_timeout)

def inspect_columns(gaia, table_names):
    records, columns = [], {}
    for name in table_names:
        identifier(name)
        table = gaia.load_table(f"{TAP_SCHEMA}.{name}", verbose=False)
        columns[name] = [str(c.name).lower() for c in table.columns]
        if "source_id" not in columns[name]:
            raise RuntimeError(f"Metadata does not advertise {name}.source_id.")
        records.append(dict(table=f"{TAP_SCHEMA}.{name}", columns=columns[name]))
    save_json(OUTPUT / "columns.json", dict(CONFIG, inspected=now(), tables=records))
    return columns

def save_provenance(path, query, rows):
    save_json(path.with_suffix(path.suffix + ".provenance.json"),
              dict(CONFIG, exported_utc=now(), query=query, rows=rows,
                   tap_cache_directory=str(OUTPUT),
                   sha256=hashlib.sha256(path.read_bytes()).hexdigest()))

## Choose a source

Keep the source ID as a decimal string or exact integer. Floating-point IDs
are rejected because they may already have lost precision.

In [ ]:
SOURCE_ID = source_id("6059400613544951552")
SOURCE_ID

## Query the source table

The session runs the known-source smoke test first. Authenticated mode always logs out.

In [ ]:
query = f"""
SELECT source_id, ra, dec, phot_g_mean_mag, bp_rp
FROM {GAIA_SOURCE_TABLE}
WHERE source_id = {SOURCE_ID}
"""
with archive_session() as gaia:
    columns = inspect_columns(gaia, ["gaia_source"])
    if not {"source_id", "ra", "dec", "phot_g_mean_mag", "bp_rp"}.issubset(columns["gaia_source"]):
        raise RuntimeError("Required columns are missing from current Gaia-source metadata.")
    result = tap(gaia, query, f"source_{SOURCE_ID}_basic_parameters")
if len(result) == 0:
    print(f"No row for source_id={SOURCE_ID} in {GAIA_SOURCE_TABLE}; source IDs can change between releases.")
else:
    display(result)

## Session cleanup

Authenticated mode closes its session in the query cell, including on failure.

In [ ]:
# No extra logout is needed after archive_session().